## Sieć neuronowa do rozpoznawania instrumentów

### Biblioteki

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score
import random
from copy import deepcopy

### Pobranie danych wejściowych z mel-spektogramu

In [2]:
main = Path("../..").resolve()
spectrograms = main / Path("spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s")
labels = spectrograms / "etykiety.csv"

data = pd.read_csv(labels)

In [3]:
print(data.shape)
print(data.columns.tolist())

(11011, 9)
['dataset', 'audio_file', 'Acoustic Guitar (steel)', 'Alto Sax', 'Bright Acoustic Piano', 'Drums', 'Electric Bass (finger)', 'Flute', 'String Ensemble 1']


In [5]:
instrumenty = [
    "Acoustic Guitar (steel)",
    "Alto Sax",
    "Bright Acoustic Piano",
    "Drums",
    "Electric Bass (finger)",
    "Flute",
    "String Ensemble 1"
]

In [26]:
for set in ["train", "validation", "test"]:
    folder = spectrograms / set
    files = list(folder.iterdir())

    print(f"\n{set}: {len(files)} elementów")
    print("Przykładowe pliki:", [p.name for p in files[:5]])


train: 7709 elementów
Przykładowe pliki: ['Track00986_32_9188.npy', 'Track01127_8_4707.npy', 'Track00849_72_10252.npy', 'Track01943_252_8239.npy', 'Track00343_100_5639.npy']

validation: 1646 elementów
Przykładowe pliki: ['Track01019_64_2140.npy', 'Track00849_72_5699.npy', 'Track01889_96_7835.npy', 'Track00343_128_9038.npy', 'Track01494_120_6376.npy']

test: 1656 elementów
Przykładowe pliki: ['Track00801_160_3815.npy', 'Track00343_192_10654.npy', 'Track02013_164_7697.npy', 'Track00343_80_8610.npy', 'Track00319_232_5610.npy']


In [ ]:
class InstrumentDataset(Dataset):
    def __init__(self, data, root, zbior):
        self.data = (
            data[data["dataset"] == zbior]
            .reset_index(drop=True)
        )
        self.root = root
        self.zbior = zbior

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        nazwa_pliku = Path(row["audio_file"]).with_suffix(".npy").name
        sciezka = self.root / self.zbior / nazwa_pliku

        spec = np.load(sciezka)
        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)

        label = row[instrumenty].to_numpy(dtype=np.float32)
        label = torch.tensor(label, dtype=torch.float32)

        return spec, label

In [ ]:
train_dataset = InstrumentDataset(data, spectrograms, "train")
val_dataset = InstrumentDataset(data, spectrograms, "validation")
test_dataset = InstrumentDataset(data, spectrograms, "test")

def create_loaders(batch_size, seed=42):
    # Tworzymy funkcję, aby sprawdzić optymalną liczbę batchy
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [9]:
train_loader, val_loader, test_loader = create_loaders(batch_size=8)

### Wybór urządzenia do obliczeń dla modelu

In [10]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    return device

### Architektura sieci

In [11]:
# jest to wstępna, jak będą dane z mel-spektogramu to się rozbuduje sieć, aby jak najlepiej zoptymalizować ją

class Instrument_CNN(nn.Module):
    def __init__(self, dropout2, dropout3):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1)
        self.conv2 = nn.Conv2d(32,64,3,1)
        self.conv3 = nn.Conv2d(64,128,3,1)
        self.dropout2 = nn.Dropout2d(dropout2)
        self.dropout3 = nn.Dropout2d(dropout3)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc3 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout3(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc3(x)

        return x


In [12]:
device = get_device()

model = Instrument_CNN(dropout2=0.2, dropout3=0.5).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Urządzenie:", device)

Urządzenie: mps


In [13]:
print("Liczba próbek:", len(train_dataset))
print("Pierwszy rekord:")
print(train_dataset.data.iloc[0])

spec, label = train_dataset[0]

print("Kształt spektrogramu:", spec.shape)
print("Kształt etykiet:", label.shape)

Liczba próbek: 7709
Pierwszy rekord:
dataset                                      train
audio_file                 Track01881_24_1982.flac
Acoustic Guitar (steel)                          0
Alto Sax                                         0
Bright Acoustic Piano                            1
Drums                                            0
Electric Bass (finger)                           1
Flute                                            0
String Ensemble 1                                0
Name: 0, dtype: object
Kształt spektrogramu: torch.Size([1, 80, 346])
Kształt etykiet: torch.Size([7])


In [14]:
X, y = next(iter(train_loader))

print("Kształt spektrogramów:", X.shape)
print("Kształt etykiet:", y.shape)
print("Przykładowe etykiety:", y[0])

Kształt spektrogramów: torch.Size([8, 1, 80, 346])
Kształt etykiet: torch.Size([8, 7])
Przykładowe etykiety: tensor([0., 1., 1., 1., 0., 0., 1.])


In [16]:
epochs = 100
patience = 10
best_epoch = 0
best_val_loss = float("inf")
epochs_without_improvement = 0
best_model = None

train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoka {epoch + 1}/{epochs} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_epoch = epoch + 1
        best_val_loss = val_loss
        best_model = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early Stopping! Brak poprawy przez {patience} epok.")
        break

model.load_state_dict(best_model)

print(f"Najlepsza epoka: {best_epoch}")
print(f"Najlepszy validation loss: {best_val_loss:.4f}")

Epoka 1/100 | Train loss: 0.5584 | Val loss: 0.5286
Epoka 2/100 | Train loss: 0.5324 | Val loss: 0.5028
Epoka 3/100 | Train loss: 0.5141 | Val loss: 0.4907
Epoka 4/100 | Train loss: 0.4990 | Val loss: 0.4522
Epoka 5/100 | Train loss: 0.4817 | Val loss: 0.4341
Epoka 6/100 | Train loss: 0.4707 | Val loss: 0.4273
Epoka 7/100 | Train loss: 0.4602 | Val loss: 0.4244
Epoka 8/100 | Train loss: 0.4499 | Val loss: 0.4108
Epoka 9/100 | Train loss: 0.4413 | Val loss: 0.3978
Epoka 10/100 | Train loss: 0.4373 | Val loss: 0.3974
Epoka 11/100 | Train loss: 0.4266 | Val loss: 0.3848
Epoka 12/100 | Train loss: 0.4245 | Val loss: 0.3908
Epoka 13/100 | Train loss: 0.4179 | Val loss: 0.3777
Epoka 14/100 | Train loss: 0.4093 | Val loss: 0.3703
Epoka 15/100 | Train loss: 0.4072 | Val loss: 0.3602
Epoka 16/100 | Train loss: 0.4003 | Val loss: 0.3485
Epoka 17/100 | Train loss: 0.3962 | Val loss: 0.3663
Epoka 18/100 | Train loss: 0.3902 | Val loss: 0.3529
Epoka 19/100 | Train loss: 0.3876 | Val loss: 0.3335
Ep

Przyjmujemy, że jeżeli prawdopodobieństwo wystąpienia instrumentu w utworze jest conajmniej $80\%$, to przyjmujemy, że instrument występuje.

In [23]:
model.eval()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)

        logits = model(X)
        probabilities = torch.sigmoid(logits).cpu().numpy()
        predictions = (probabilities >= 0.8).astype(int)
        true_labels = y.cpu().numpy().astype(int)

        print("Prawdopodobieństwa:")
        display(pd.DataFrame(probabilities, columns=instrumenty).round(3))

        print("Przewidywane etykiety:")
        display(pd.DataFrame(predictions, columns=instrumenty))

        print("Prawdziwe etykiety:")
        display(pd.DataFrame(true_labels, columns=instrumenty))

        break

Prawdopodobieństwa:


,Acoustic Guitar (steel),Alto Sax,Bright Acoustic Piano,Drums,Electric Bass (finger),Flute,String Ensemble 1
0,0.631,0.440,0.950,1.0,0.999,0.462,0.921
1,0.638,0.273,0.999,1.0,0.014,0.183,0.145
2,0.015,1.000,0.003,1.0,0.004,0.972,0.009
3,0.088,0.421,0.611,1.0,0.012,0.371,0.810
4,0.993,0.273,0.964,1.0,0.955,0.738,0.129
5,1.000,0.339,0.062,0.0,0.001,0.065,1.000
6,0.011,0.200,0.999,0.0,1.000,0.427,0.378
7,0.004,0.923,0.990,1.0,0.000,0.938,0.941


Przewidywane etykiety:


,Acoustic Guitar (steel),Alto Sax,Bright Acoustic Piano,Drums,Electric Bass (finger),Flute,String Ensemble 1
0,0,0,1,1,1,0,1
1,0,0,1,1,0,0,0
2,0,1,0,1,0,1,0
3,0,0,0,1,0,0,1
4,1,0,1,1,1,0,0
5,1,0,0,0,0,0,1
6,0,0,1,0,1,0,0
7,0,1,1,1,0,1,1


Prawdziwe etykiety:


,Acoustic Guitar (steel),Alto Sax,Bright Acoustic Piano,Drums,Electric Bass (finger),Flute,String Ensemble 1
0,1,0,1,1,1,1,1
1,0,1,1,1,0,0,0
2,0,1,0,1,0,1,0
3,0,1,0,1,0,0,1
4,1,0,1,1,1,1,0
5,1,0,0,0,0,0,1
6,0,0,1,0,1,0,1
7,0,1,1,1,0,1,1


In [ ]:
torch.save(model.state_dict(), "model_weights.pth")
model.eval()

all_predictions = []
all_labels = []
test_loss = 0.0

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        loss = criterion(logits, y)

        test_loss += loss.item() * X.size(0)

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.8).float()

        all_predictions.append(predictions.cpu().numpy())
        all_labels.append(y.cpu().numpy())

test_loss /= len(test_loader.dataset)

all_predictions = np.concatenate(all_predictions, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

f1_micro = f1_score(all_labels, all_predictions, average="micro", zero_division=0)
f1_macro = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
f1_per_instrument = f1_score(all_labels, all_predictions, average=None, zero_division=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Micro-F1 (miara F1 dla wszystkich etykiet): {f1_micro:.4f}")
print(f"Macro-F1 (średnia miara F1 dla instrumentów): {f1_macro:.4f}")

results = pd.DataFrame({"Instrument": instrumenty, "F1-score": f1_per_instrument})

display(results.round(4))

Test loss: 0.2277
Micro-F1 (miara F1 dla wszystkich etykiet): 0.8315
Macro-F1 (średnia miara F1 dla instrumentów): 0.8024


,Instrument,F1-score
0,Acoustic Guitar (steel),0.8676
1,Alto Sax,0.6415
2,Bright Acoustic Piano,0.8138
3,Drums,0.9928
4,Electric Bass (finger),0.9372
5,Flute,0.4852
6,String Ensemble 1,0.8785
